# Production ML Monitoring, Calibration and Safe Rollout

**Course alignment:** Data Science in Production (Week 12, Session 1)  
**Type:** Original supplementary applied notebook  
**Scenario:** A model supporting multiple hospital sites

This notebook operationalises four monitoring layers: system, data, model and business. It demonstrates training-serving skew, feature freshness, schema validation, distribution drift, calibration, delayed labels, shadow deployment and canary promotion. All data are synthetic. The hospital example is an engineering exercise, not a medical tool or clinical recommendation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp, chi2_contingency
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss, accuracy_score
from sklearn.calibration import calibration_curve

rng=np.random.default_rng(44)
X,y=make_classification(
    n_samples=7000,n_features=8,n_informative=5,n_redundant=1,
    weights=[.88,.12],class_sep=1.0,random_state=44
)
columns=[f"feature_{i}" for i in range(X.shape[1])]
df=pd.DataFrame(X,columns=columns)
df["label"]=y
df["hospital"]=rng.choice(["A","B","C"],len(df),p=[.5,.3,.2])
train,holdout=train_test_split(df,test_size=.25,random_state=42,stratify=df["label"])
features=columns
model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=2000,class_weight="balanced"))
model.fit(train[features],train["label"])
offline_prob=model.predict_proba(holdout[features])[:,1]
offline_pred=(offline_prob>=.5).astype(int)
print("Offline ROC-AUC:",round(roc_auc_score(holdout["label"],offline_prob),3))
print("Offline accuracy:",round(accuracy_score(holdout["label"],offline_pred),3))
print("Offline Brier:",round(brier_score_loss(holdout["label"],offline_prob),3))

## 1. Training-serving skew and feature drift

A production feed is simulated with shifted feature distributions, a changed category mix and missing values. Compare the training reference with the serving window. Statistical tests should be interpreted with sample size and operational context; a small p-value alone does not establish material harm.

In [ ]:
live=rng.normal(loc=.45,scale=1.25,size=(2200,len(features)))
live=pd.DataFrame(live,columns=features)
live["hospital"]=rng.choice(["A","B","C"],len(live),p=[.2,.35,.45])
# Simulate one feature becoming unavailable in some requests.
live.loc[rng.choice(live.index,180,replace=False),"feature_2"]=np.nan

drift_rows=[]
for col in features:
    reference=train[col].dropna()
    current=live[col].dropna()
    test=ks_2samp(reference,current)
    drift_rows.append({
        "feature":col,
        "train_mean":reference.mean(),
        "live_mean":current.mean(),
        "train_missing_rate":train[col].isna().mean(),
        "live_missing_rate":live[col].isna().mean(),
        "ks_statistic":test.statistic,
        "ks_p_value":test.pvalue
    })
drift=pd.DataFrame(drift_rows)
display(drift.round(4))

## 2. Population Stability Index (PSI) as a descriptive drift indicator

PSI compares binned reference and current distributions. Bin edges should be defined from the reference population and kept fixed. Rules of thumb used in industry vary; thresholds must be calibrated to the feature and business risk rather than treated as universal laws.

In [ ]:
def calculate_psi(reference, current, bins=10, eps=1e-6):
    ref=np.asarray(pd.Series(reference).dropna())
    cur=np.asarray(pd.Series(current).dropna())
    edges=np.unique(np.quantile(ref,np.linspace(0,1,bins+1)))
    if len(edges)<3:
        return np.nan
    edges[0],edges[-1]=-np.inf,np.inf
    ref_counts=np.histogram(ref,bins=edges)[0]
    cur_counts=np.histogram(cur,bins=edges)[0]
    ref_pct=np.clip(ref_counts/ref_counts.sum(),eps,None)
    cur_pct=np.clip(cur_counts/cur_counts.sum(),eps,None)
    return np.sum((cur_pct-ref_pct)*np.log(cur_pct/ref_pct))

psi_table=pd.DataFrame({
    "feature":features,
    "PSI":[calculate_psi(train[c],live[c]) for c in features]
}).sort_values("PSI",ascending=False)
display(psi_table.round(3))

## 3. Schema, null-rate and feature freshness checks

Schema validation catches changes such as an unexpected type, renamed column or missing required field. Feature freshness measures the age of a value when the model consumes it. A feature can have plausible values and still be stale.

In [ ]:
expected_schema={
    "feature_0":"float64","feature_1":"float64","feature_2":"float64",
    "feature_3":"float64","feature_4":"float64","feature_5":"float64",
    "feature_6":"float64","feature_7":"float64"
}
schema_report=pd.DataFrame([
    {"feature":c,"expected_type":expected_schema[c],
     "observed_type":str(live[c].dtype),
     "type_matches":str(live[c].dtype)==expected_schema[c],
     "null_rate":live[c].isna().mean()}
    for c in features
])
display(schema_report)

now=pd.Timestamp("2026-11-01 12:00:00")
feature_updates=now-pd.to_timedelta(rng.integers(0,90,5000),unit="m")
freshness_minutes=(now-feature_updates).total_seconds()/60
print("Freshness minutes p50:",np.percentile(freshness_minutes,50))
print("Freshness minutes p95:",np.percentile(freshness_minutes,95))
print("Share older than 30 minutes:",(freshness_minutes>30).mean())

## 4. System monitoring: latency, throughput, queue depth and errors

Average latency can hide tail risk. P95 and P99 describe the slower requests; queue depth can indicate accumulating work before latency or timeouts become widespread.

In [ ]:
requests=12000
latency_ms=rng.lognormal(mean=4.4,sigma=.85,size=requests)
# Simulate a short overload window with longer tail latency.
overload=rng.choice(np.arange(requests),size=800,replace=False)
latency_ms[overload]*=4
status_ok=rng.random(requests)>.018
queue_depth=np.maximum(0,np.cumsum(rng.normal(0,1.4,requests))).astype(int)
system_metrics=pd.Series({
    "P50 latency ms":np.percentile(latency_ms,50),
    "P95 latency ms":np.percentile(latency_ms,95),
    "P99 latency ms":np.percentile(latency_ms,99),
    "Error rate":1-status_ok.mean(),
    "Mean queue depth":queue_depth.mean(),
    "Maximum queue depth":queue_depth.max(),
})
display(system_metrics.to_frame("value").round(3))

fig,ax=plt.subplots(figsize=(11,4))
ax.plot(queue_depth[:1200],linewidth=.8)
ax.set(title="Example request queue depth",xlabel="Request sequence",ylabel="Waiting requests")
ax.grid(alpha=.2)
plt.tight_layout()
plt.show()

## 5. Model monitoring: confidence versus correctness and calibration

High average confidence is not proof of correctness. Calibration compares predicted probabilities with observed event frequencies. Here the holdout labels are available, so a reliability curve can be calculated. In many production systems, labels arrive later; monitoring must explicitly account for that delay.

In [ ]:
fraction_positive,mean_predicted=calibration_curve(
    holdout["label"],offline_prob,n_bins=10,strategy="quantile"
)
fig,ax=plt.subplots(figsize=(6,5))
ax.plot(mean_predicted,fraction_positive,marker="o",label="Model")
ax.plot([0,1],[0,1],linestyle="--",label="Perfect calibration")
ax.set(title="Reliability curve",xlabel="Mean predicted probability",ylabel="Observed positive fraction")
ax.legend()
ax.grid(alpha=.2)
plt.tight_layout()
plt.show()

print("Mean predicted probability:",round(offline_prob.mean(),3))
print("Observed event rate:",round(holdout["label"].mean(),3))
print("Brier score:",round(brier_score_loss(holdout["label"],offline_prob),4))

## 6. Segment-level monitoring

Aggregate metrics can hide a failing site or subgroup. Compare performance by hospital in the labelled holdout. In real deployments, ensure sample sizes and privacy protections are sufficient before publishing subgroup results.

In [ ]:
segmented=holdout[["hospital","label"]].copy()
segmented["probability"]=offline_prob
segmented["prediction"]=offline_pred
site_rows=[]
for site,part in segmented.groupby("hospital"):
    site_rows.append({
        "hospital":site,
        "n":len(part),
        "event_rate":part["label"].mean(),
        "roc_auc":roc_auc_score(part["label"],part["probability"]) if part["label"].nunique()>1 else np.nan,
        "brier":brier_score_loss(part["label"],part["probability"]),
        "recall_at_0_5":((part["prediction"]==1)&(part["label"]==1)).sum()/max(1,(part["label"]==1).sum())
    })
display(pd.DataFrame(site_rows).round(3))

## 7. Shadow and canary rollout decision

In shadow mode, a candidate model receives copies of live inputs but does not control user-facing decisions. A canary directs a small share of real traffic to the candidate while retaining the established model for the remainder. Define promotion criteria and rollback rules before deployment.

In [ ]:
# Synthetic example of predeclared release gates.
shadow={
    "schema_valid":True,
    "feature_freshness_p95_minutes":12.0,
    "candidate_brier":0.091,
    "baseline_brier":0.096,
    "candidate_p95_latency_ms":180,
    "baseline_p95_latency_ms":165,
    "critical_site_recall_drop":0.01
}
promotion_checks={
    "schema_valid":shadow["schema_valid"],
    "freshness_under_20m":shadow["feature_freshness_p95_minutes"]<20,
    "no_brier_regression":shadow["candidate_brier"]<=shadow["baseline_brier"],
    "latency_under_250ms":shadow["candidate_p95_latency_ms"]<250,
    "site_recall_drop_under_3pp":shadow["critical_site_recall_drop"]<.03
}
display(pd.Series(promotion_checks,name="passed"))
promote=all(promotion_checks.values())
print("Candidate eligible for limited canary:",promote)

canary_traffic_fraction=.05
rollback={
    "error_rate_above_2pct":.025>.02,
    "p95_latency_over_500ms":430>500,
    "critical_site_recall_drop_over_5pp":.06>.05,
    "schema_validation_failure":False
}
display(pd.Series(rollback,name="rollback_trigger"))
print("Rollback required:",any(rollback.values()))

## 8. Build a four-layer monitoring plan

| Layer | Example metric | Why it matters |
|---|---|---|
| System | P95/P99 latency, error rate, queue depth, CPU/memory | Service reliability and capacity |
| Data | Null rate, schema version, distribution shift, feature freshness | Detect bad or stale inputs |
| Model | Prediction distribution, calibration, recall, drift | Detect degraded model behavior |
| Business/clinical | Alert acknowledgement, useful alert rate, site-level outcomes | Connect model outputs to real outcomes |

**Exercise:** For each metric, propose a baseline, alert threshold, owner and response action. Avoid setting a threshold without a known operating range and acceptable risk.

## Limitations

This is an engineering simulation. It does not define a clinically safe alert threshold, replace hospital governance, or prove that a model is suitable for healthcare. In real systems, delayed labels, privacy, site heterogeneity, human workflow and clinical safety review are essential.